In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """多头注意力机制"""
    def __init__(self, d_model, nhead, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.d_k = d_model // nhead
        
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.out_linear = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # 线性变换并分头
        q = self.q_linear(q).view(batch_size, -1, self.nhead, self.d_k).transpose(1,2)
        k = self.k_linear(k).view(batch_size, -1, self.nhead, self.d_k).transpose(1,2)
        v = self.v_linear(v).view(batch_size, -1, self.nhead, self.d_k).transpose(1,2)
        
        # 计算注意力
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        # 合并多头
        output = torch.matmul(attn, v)
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.out_linear(output)

class FeedForward(nn.Module):
    """前馈网络"""
    def __init__(self, d_model, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        
    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = self.dropout(x)
        return self.linear2(x)

class EncoderLayer(nn.Module):
    """编码器层"""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.ffn = FeedForward(d_model, dim_feedforward, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, src, src_mask=None):
        # 自注意力
        src2 = self.self_attn(src, src, src, src_mask)
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        
        # 前馈网络
        src2 = self.ffn(src)
        src = src + self.dropout2(src2)
        return self.norm2(src)

class DecoderLayer(nn.Module):
    """解码器层"""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.cross_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.ffn = FeedForward(d_model, dim_feedforward, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        # 自注意力（带掩码）
        tgt2 = self.self_attn(tgt, tgt, tgt, tgt_mask)
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        
        # 交叉注意力（编码器输出作为k,v）
        tgt2 = self.cross_attn(tgt, memory, memory, memory_mask)
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        
        # 前馈网络
        tgt2 = self.ffn(tgt)
        tgt = tgt + self.dropout3(tgt2)
        return self.norm3(tgt)

class TransformerEncoder(nn.Module):
    """完整编码器"""
    def __init__(self, num_layers, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        
    def forward(self, src, src_mask=None):
        for layer in self.layers:
            src = layer(src, src_mask)
        return src

class TransformerDecoder(nn.Module):
    """完整解码器"""
    def __init__(self, num_layers, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        for layer in self.layers:
            tgt = layer(tgt, memory, tgt_mask, memory_mask)
        return tgt

In [3]:
import torch
import torch.nn as nn
import json
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
import math
import os

class QADataset(Dataset):
    """处理 SQuAD 格式数据为模型输入格式"""
    def __init__(self, file_path, tokenizer, max_length=256):
        self.data = []
        with open(file_path, 'r') as f:
            squad = json.load(f)

        for article in squad['data'][:200]:
            for para in article['paragraphs']:
                context = para['context']
                for qa in para['qas']:
                    question = qa['question']
                    answer = qa['answers'][0]['text'] if qa['answers'] else ""
                    input_text = f"{question} context: {context}"
                    output_text = answer

                    inputs = tokenizer(input_text, max_length=max_length, padding='max_length', truncation=True, return_tensors="pt")
                    targets = tokenizer(output_text, max_length=max_length, padding='max_length', truncation=True, return_tensors="pt")

                    self.data.append({
                        'input_ids': inputs['input_ids'].squeeze(0),
                        'attention_mask': inputs['attention_mask'].squeeze(0),
                        'labels': targets['input_ids'].squeeze(0)
                    })

        self.data = self.data[:1]  # 可根据需求调节数据量

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


# ------------------ 位置编码 ------------------ #
class PositionalEncoding(nn.Module):
    """正弦位置编码"""
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pos = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)

        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1)]
        return self.dropout(x)


# ------------------ 问答Transformer模型 ------------------ #
class TransformerQA(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        self.encoder = TransformerEncoder(num_encoder_layers, d_model, nhead, dim_feedforward, dropout)
        self.decoder = TransformerDecoder(num_decoder_layers, d_model, nhead, dim_feedforward, dropout)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        src = self.pos_encoder(self.embedding(src) * math.sqrt(self.d_model))
        memory = self.encoder(src, src_mask)

        tgt = self.pos_encoder(self.embedding(tgt) * math.sqrt(self.d_model))
        output = self.decoder(tgt, memory, tgt_mask)

        return self.fc_out(output)

    def generate_mask(self, src, tgt):
        src_pad_mask = (src == 0).unsqueeze(1).unsqueeze(2)  # [B, 1, 1, S]
        tgt_pad_mask = (tgt == 0).unsqueeze(1).unsqueeze(2)
        seq_len = tgt.size(1)
        tgt_sub_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(src.device)
        tgt_mask = tgt_pad_mask | tgt_sub_mask
        return src_pad_mask, tgt_mask

    def generate_answer(self, src_ids, tokenizer, max_len=50):
        """自回归生成答案"""
        self.eval()
        device = src_ids.device

        src_mask = (src_ids == tokenizer.pad_token_id).unsqueeze(1).unsqueeze(2)
        memory = self.encoder(self.pos_encoder(self.embedding(src_ids) * math.sqrt(self.d_model)), src_mask)

        outputs = torch.LongTensor([[tokenizer.cls_token_id]]).to(device)

        for _ in range(max_len):
            _, tgt_mask = self.generate_mask(src_ids, outputs)
            tgt_emb = self.pos_encoder(self.embedding(outputs) * math.sqrt(self.d_model))
            dec_out = self.decoder(tgt_emb, memory, tgt_mask)
            logits = self.fc_out(dec_out)[:, -1, :]

            # 温度采样 + Top-k策略
            logits /= 10
            topk_vals, topk_idx = logits.topk(10)
            probs = torch.softmax(topk_vals, dim=-1)
            next_token = topk_idx.gather(-1, torch.multinomial(probs, num_samples=1))

            outputs = torch.cat([outputs, next_token], dim=1)
            if next_token.item() == tokenizer.sep_token_id:
                break

        return tokenizer.decode(outputs.squeeze(0), skip_special_tokens=True)

In [4]:
def chat(model, tokenizer, device, max_len=50):
    model.eval()
    print("进入问答模式，输入 'exit' 退出。")
    while True:
        question = input("用户: ")
        if question.strip().lower() == 'exit':
            print("结束对话。")
            break
        inputs = tokenizer(question, max_length=256, padding='max_length', truncation=True, return_tensors="pt")
        src = inputs['input_ids'].to(device)
        answer = model.generate_answer(src, tokenizer, max_len)
        print(f"模型: {answer}")



tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerQA(vocab_size=tokenizer.vocab_size)
ckpt_path = '/kaggle/input/model_epoch_9/pytorch/default/1/model_epoch_9.pth'
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.to(device)

chat(model, tokenizer, device)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

/tmp/ipykernel_31/3090821918.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(ckpt_path, map_location=device))


...

用户:  hello


模型: of worldsat cu. world cansat. cu lights bra lights cu cu. of disease ofsat. cu e. thatsat of that ofro of that cu that braro that. that that of can worldro.. that that can cu


用户:  who are you


模型: of bra that bra bra to brajmde of championship foundro bra of bra of cude israeliro of using brajm israeli bra that using cu using braro cu thatde von that von cu to bra thatde israeli that cu of brade


KeyboardInterrupt: Interrupted by user